# Example of Exploratory Data Analysis and ML Modeling

In this notebook, we will walk through the key stages of a Data Science workflow using a housing dataset from King County. You have already seen some of the early steps in a previous lecture. This time, we will go further by building and evaluating machine learning models.

As a quick reminder, here is the Data Science lifecycle:

<center><img src="./images/DS_LC.png" alt="drawing" width="500"/></center>

The goal is to implement the essential steps of a supervised machine learning workflow with the Python library scikit-learn.  

We will use the following methods:  
- feature engineering,
- train/test split,
- model training,
- hyperparameter optimization with grid search and cross-validation,
- and model evaluation.

To make the workflow more concrete, we will use the following fictional scenario:

**You have just landed your dream job as a data scientist at an innovative company near Seattle. Before you move, you want to understand the local housing market and estimate fair property values so you can negotiate with confidence.**

```mermaid
flowchart LR
    A[Raw housing data] --> B[Data preparation]
    B --> C[Exploratory data analysis]
    C --> D[Feature engineering]
    D --> E[Model training]
    E --> F[Evaluation and tuning]
    F --> G[Save the final model]
```


![](images/seattle_skyline.jpg)
(<span>Photo by <a href="https://unsplash.com/@phoebezzf?utm_source=unsplash&amp;utm_medium=referral&amp;utm_content=creditCopyText">Zhifei Zhou</a> on <a href="https://unsplash.com/s/photos/seattle-skyline?utm_source=unsplash&amp;utm_medium=referral&amp;utm_content=creditCopyText">Unsplash</a></span>)

**Rather than relying on intuition alone, we will use data to support the house-hunting process. Our aim is to predict home prices in and around Seattle so that we can estimate a realistic property value before negotiating.**

## Business Understanding

Seattle is located in King County, which is shown on the map below. To keep commuting time manageable, we limit the property search to this area.



![](images/king_county_districts.jpeg)

(Photo from King County [imap](https://gismaps.kingcounty.gov/iMap/))

For this project, we use a dataset that contains property sales in King County from May 2014 to May 2015.

Before building any model, it is important to understand what information the dataset contains. The table below summarizes the available columns:

| **Column name** | **Description** |
|---|---|
| `id` | Unique identifier for a house sale |
| `date` | Date of sale |
| `price` | Sale price of the house |
| `bedrooms` | Number of bedrooms |
| `bathrooms` | Number of bathrooms; `.5` indicates a toilet room without a shower |
| `sqft_living` | Interior living area in square feet |
| `sqft_lot` | Lot size in square feet |
| `floors` | Number of floors |
| `waterfront` | Whether the house has a waterfront view (`0` = no, `1` = yes) |
| `view` | Index from `0` to `4` indicating the quality of the view |
| `condition` | Index from `1` to `5` describing the condition of the property |
| `grade` | Index from `1` to `13` describing construction and design quality |
| `sqft_basement` | Basement area in square feet |
| `yr_built` | Year the house was built |
| `yr_renovated` | Year of the last renovation |
| `zipcode` | ZIP code of the property |
| `lat` | Latitude |
| `long` | Longitude |
| `sqft_living15` | Interior living area of the nearest 15 neighbors |
| `sqft_lot15` | Lot size of the nearest 15 neighbors |



## Importing the Libraries

First, we import the libraries we will need throughout the notebook.



In [ ]:
# Basic imports
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV

In [ ]:
# Load the dataset with pandas
kc_data = pd.read_csv("data/King_County_House_prices_dataset.csv")

In [ ]:
# Look at the first 5 rows of the dataset.
# This helps us confirm that the file was loaded correctly.
kc_data.head(10)

## Data Preparation

After loading the data, we need a reliable overview of its quality and structure.
We will answer the following questions:

- How many house sales are included in the dataset?
- What data types are present (`int`, `float`, `object`)?
- Are there any missing values?
- How many distinct values does each variable contain?



**1.** How many house sales are included in the dataset?



In [ ]:
kc_data.shape

Each row represents one house sale.

The dataset contains 21597 sales records.



**2.** What data types are present (`int`, `float`, `object`)?
Which values look surprising?


In [ ]:
kc_data.info()

Our dataset contains 8 floating-point columns, 11 integer columns, and 2 object columns.
We would normally expect variables such as sizes, room counts, indexes, and geographic information to be stored as numeric values.

The `date` column is stored as text, which is reasonable at this stage, but `sqft_basement` is also stored as text. That is surprising.
When we inspect the data more closely, we can see why: the column contains a `?` value, so pandas cannot safely interpret the full column as numeric.



**3.** Are there any missing values?

We have missing values in the columns `waterfront`, `view`, and `yr_renovated`.
The largest share of missing values appears in `yr_renovated`.

We will need to decide how to handle each of these columns carefully, because different kinds of missing values call for different strategies.



**4.** How many distinct values does each variable contain?



In [ ]:
kc_data.nunique()

Interesting findings:

**1.** We have 21420 distinct house IDs, which suggests that up to 177 houses may have been sold more than once.

**2.** We observe 3622 distinct sale prices, so many properties were sold for the same price.

**3.** Although `grade` is defined on a scale from 1 to 13, only 11 distinct values appear in the dataset.

**4.** The houses in our dataset were built across 116 different years.



Next, we can look at the distribution of the variables in more detail.

`kc_data.describe()` reports the following summary statistics:
- **count**: number of non-missing values
- **mean**: arithmetic average
- **std**: standard deviation
- **min**: smallest observed value
- **25%**: first quartile
- **50%**: median
- **75%**: third quartile
- **max**: largest observed value

These values help us spot outliers, skewed distributions, and variables that may need additional cleaning or transformation.



In [ ]:
kc_data.describe().round(2)

Interesting findings:

**1.** The average sale price is $540,296. The most expensive house sold for $7,700,000, while the cheapest sold for $78,000.

**2.** Half of the houses have 3 bedrooms or fewer. The maximum value is 33 bedrooms, which immediately looks suspicious.

**3.** On average, each house has about 2.1 bathrooms per bedroom.

**4.** 75% of houses have 2 floors or fewer.

**5.** Only about 1% of the houses have a waterfront view. Because `waterfront` is binary, its mean can be interpreted as a proportion.

**6.** The oldest house in the dataset dates from 1900, while the newest dates from 2015.



We could extract many more insights from the summary table, but we have already identified a few issues that should be handled during data cleaning:

- missing values
- the `?` entry in `sqft_basement`
- the record with 33 bedrooms

The remaining distributions will be easier to interpret later with charts. For now, we will focus on cleaning the data.



**33 bedrooms**

The maximum bedroom count points to a property with 33 bedrooms but only 2 bathrooms.



In [ ]:
# Display the row where the condition `bedrooms == 33` is true.
kc_data.query("bedrooms == 33")

A house with 33 bedrooms, 2 bathrooms, and only 1,620 square feet is extremely unlikely. This record is probably a data-entry error, so we remove it from the dataset.



In [ ]:
# Drop the implausible row.
kc_data.drop(15856, axis=0, inplace=True)

**`?` in `sqft_basement`**

From the output of `.info()`, we can see that `sqft_basement` is stored as an object rather than a numeric value. The reason is that at least one row contains `?` instead of a number.

We will reconstruct the basement size as `sqft_living - sqft_above`.



In [ ]:
# Recalculate `sqft_basement` as `sqft_living - sqft_above`.
kc_data["sqft_basement"] = kc_data["sqft_living"] - kc_data["sqft_above"]

**Missing values**


In [ ]:
# Count missing values and calculate their percentage.
missing_values = kc_data.isnull().sum().to_frame(name="count")
missing_values["percentage"] = (missing_values["count"] / kc_data.shape[0] * 100).round(
    2
)
missing_values.query("count != 0")

After replacing the invalid basement entries, three variables still contain missing values:

- `waterfront`: 2376 missing values (11.00%)
- `view`: 63 missing values (0.29%)
- `yr_renovated`: 3842 missing values (17.79%)

The `view` column has very few missing values, and the value 0 is by far the most common category. Because of that strong imbalance, replacing the missing values with 0 is a reasonable simplification here.



In [ ]:
# Display the frequency of each value.
kc_data["view"].value_counts()

In [ ]:
# Replace missing values in `view` with the most frequent value (0).
kc_data["view"] = kc_data["view"].fillna(0)

The `waterfront` column shows a similar pattern, so we again replace the missing values with 0.



In [ ]:
# Display the frequency of each value.
kc_data["waterfront"].value_counts()

In [ ]:
# Replace missing values in `waterfront` with the most frequent value (0).
kc_data["waterfront"] = kc_data["waterfront"].fillna(0)

In [ ]:
# Inspect the missing data again.
missing_values = kc_data.isnull().sum().to_frame(name="count")
missing_values["percentage"] = missing_values["count"] / kc_data.shape[0] * 100
missing_values.query("count != 0")

Because `yr_renovated` has a relatively large number of missing values, we do not simply fill it with a single constant. Instead, we combine `yr_built` and `yr_renovated` into a new feature called `last_known_change`, then remove the original two columns.



In [ ]:
# Create an empty list to store the derived values.
last_known_change = []

# Inspect the `yr_renovated` value for each row.
for idx, yr_re in kc_data["yr_renovated"].items():
    # If `yr_renovated` is 0 or missing, use the original build year instead.
    if str(yr_re) == "nan" or yr_re == 0.0:
        last_known_change.append(kc_data["yr_built"][idx])
    # Otherwise, keep the renovation year in the new list.
    else:
        last_known_change.append(int(yr_re))

In [ ]:
# Create a new column from the values collected above.
kc_data["last_known_change"] = last_known_change

In [ ]:
# Drop the original `yr_renovated` and `yr_built` columns.
kc_data.drop("yr_renovated", axis=1, inplace=True)
kc_data.drop("yr_built", axis=1, inplace=True)

At this point, the main data cleaning steps are complete.

Commands such as `.describe()` already gave us a quick statistical overview, but exploratory data analysis becomes much easier once we visualize the distributions directly.



## Exploratory Data Analysis

Now we move from cleaning to exploration.
Histograms are a useful first step because they show how each variable is distributed and whether a variable behaves more like a continuous measure or a category.



In [ ]:
# Select variables for a closer visual inspection.
columns_histogram = [
    "price",
    "bathrooms",
    "bedrooms",
    "floors",
    "grade",
    "last_known_change",
    "sqft_living",
    "sqft_lot",
]

In [ ]:
# Plot histograms.
kc_data[columns_histogram].hist(bins=50, figsize=(20, 15))
plt.show()

Histograms help us understand both the overall distribution and the rough shape of each variable.

Findings:
- `price` is right-skewed, with a small number of very expensive houses stretching the distribution.
- More than 5000 houses have 2.5 bathrooms. In the U.S., `.5` typically indicates a toilet room without a shower.
- Most houses have between 2 and 5 bedrooms, with 3 bedrooms being the most common.
- Most houses have only one floor.
- Most houses have a grade between 6 and 10 on the 1 to 13 scale.
- Very few houses were built or renovated between 1930 and 1940. Construction increases noticeably after that period.
- `sqft_living` is also somewhat right-skewed.
- Extreme lot sizes distort the lot-area histogram, which makes the main distribution harder to see.

Another useful view is the box plot, which displays the same summary statistics graphically.



In [ ]:
# Plot box plots.
fig = px.box(kc_data, y="price", labels={"price": "House Price in $"})
fig.show()

If we try the box plot with different columns, we can clearly see outliers in several variables.  

Outliers matter because they can strongly affect model quality, but they should not be removed automatically.

For the property with 33 bedrooms, the value was implausible enough to treat it as an error and remove the row.
By contrast, very expensive houses are rare but believable in the Seattle area, so we keep them.

This is a good example of why domain knowledge matters: **an unusual value is not always a wrong value**.



It is also useful to inspect relationships between variables.
A pair plot gives a quick overview by plotting each variable against the others and showing the univariate distributions on the diagonal.

When reading the pair plot, we are especially interested in possible relationships such as:
- how price changes as living area increases
- whether better grades are associated with higher prices
- whether some features appear to move together



In [ ]:
# Plot scatterplots.
sns.pairplot(kc_data[columns_histogram]);

A heatmap provides another view of correlation, this time by summarizing the linear relationship between every pair of variables.

Strong red tones indicate positive correlation, while strong blue tones indicate negative correlation. For example:
- larger living area is often associated with higher price
- greater distance from the wealth center may be associated with lower price

**Correlation does not prove causation**, but it helps us identify promising variables for a linear model.



In [ ]:
# Heatmap of the Pearson correlation coefficients
numeric_kc_data = kc_data.select_dtypes(include=["number"])
mask = np.triu(numeric_kc_data.corr())
plt.figure(figsize=(20, 15))
ax = sns.heatmap(round(numeric_kc_data.corr(), 2), annot=True, mask=mask, cmap="RdBu_r")

## Feature Engineering

Feature engineering means creating new variables that may explain the target more clearly or improve predictive performance.

In this project, we first adjust the target variable itself. Buyers often think in terms of value for money rather than total price alone, so we use **price per square foot of living space** as the quantity to model.



In [ ]:
# Create a variable for price per square foot of living space.
kc_data["sqft_price"] = (
    kc_data.price / (kc_data.sqft_living + kc_data.sqft_lot)
).round(2)

**Distance to the center of wealth**

One possible explanatory feature is the distance to a highly desirable area. As a simple proxy, we use Bill Gates' residence in Medina, on Lake Washington, as a rough "center of wealth".
According to Wikipedia, the estate has an estimated value of about **US$147.5 million**.

<p><a href="https://commons.wikimedia.org/wiki/File:Residence_of_Bill_Gates.jpg#/media/File:Residence_of_Bill_Gates.jpg"><img src="https://upload.wikimedia.org/wikipedia/commons/thumb/2/26/Residence_of_Bill_Gates.jpg/1200px-Residence_of_Bill_Gates.jpg" alt="Bill Gates residence"></a></p>

To create this feature, we compute the geographic distance between each house and that reference point.
We correct longitude by `cos(lat)` because the physical distance represented by one degree of longitude depends on latitude.
This gives us a more realistic distance in kilometers.



In [ ]:
kc_data[kc_data["sqft_price"] == kc_data["sqft_price"].max()]

In [ ]:
kc_data[kc_data["price"] == kc_data["price"].max()]

In [ ]:
# Absolute difference in latitude between the center and the property.
kc_data["delta_lat"] = np.absolute(47.62774 - kc_data["lat"])
# Absolute difference in longitude between the center and the property.
kc_data["delta_long"] = np.absolute(-122.24194 - kc_data["long"])
# Distance between the center and the property.
kc_data["center_distance"] = (
    (
        (kc_data["delta_long"] * np.cos(np.radians(47.6219))) ** 2
        + kc_data["delta_lat"] ** 2
    )
    ** (1 / 2)
    * 2
    * np.pi
    * 6378
    / 360
)

Now let us check whether this assumption is supported by the data: are properties closer to the wealth center generally more expensive?



In [ ]:
# Scatter plot: price versus distance to the center.
sns.relplot(y="price", x="center_distance", data=kc_data);

In [ ]:
# Scatter plot: price per square foot versus distance to the center.
sns.relplot(y="sqft_price", x="center_distance", data=kc_data);

The charts support our hypothesis: price tends to decrease as distance from the wealth center increases.
The price-performance ratio also improves with distance, because the price per square foot tends to fall.

We can also plot all houses on a map of King County.
The color indicates how close each house is to the wealth center, while the marker size reflects sale price.



In [ ]:
fig = px.scatter_map(
    kc_data,
    lat="lat",
    lon="long",
    hover_name="id",
    hover_data=["sqft_price", "sqft_living", "zipcode", "floors"],
    size="sqft_price",
    color="center_distance",
    color_continuous_scale=["green", "yellow", "red"],
    color_discrete_sequence=["fuchsia"],
    zoom=7.7,
    height=400,
)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

**Distance to the waterfront**

Next, we construct a rough measure of water proximity.
We first extract all houses with `waterfront = 1` into a list called `water_list`. Then, for each house in the dataset, we calculate the minimum distance to any house in that list.

This is only an approximation, because waterfront houses do not describe the entire shoreline perfectly, but it still gives us a useful proxy for how close a property is to the water.



In [ ]:
# This helper function calculates the distance between one house and a reference location.
def dist(long, lat, ref_long, ref_lat):
    """dist computes the distance in km to a reference location.
    Input: long and lat of the location of interest and ref_long and ref_lat
    as the long and lat of the reference location"""
    delta_long = long - ref_long
    delta_lat = lat - ref_lat
    delta_long_corr = delta_long * np.cos(np.radians(ref_lat))
    return (
        ((delta_long_corr) ** 2 + (delta_lat) ** 2) ** (1 / 2) * 2 * np.pi * 6378 / 360
    )

In [ ]:
# Add all waterfront houses to the reference list.
water_list = kc_data.query("waterfront == 1")
water_list.head()

In [ ]:
water_distance = []
# For each row, calculate the distance to the nearest waterfront house.
for idx, lat in kc_data["lat"].items():
    ref_list = []
    for x, y in zip(list(water_list["long"]), list(water_list["lat"])):
        ref_list.append(dist(kc_data["long"][idx], kc_data["lat"][idx], x, y).min())
    water_distance.append(min(ref_list))

In [ ]:
# Create a new column from the previously computed list.
kc_data["water_distance"] = water_distance

In [ ]:
# Create a new column from the previously computed list.
kc_data.describe().round(2)

The new variables are appended to the dataset.
On average, the houses in our dataset are about 17 km from the wealth center and almost 19 km from a house with a waterfront view. The house farthest from the wealth center is roughly 70 km away.

Again, we can test whether proximity to the water appears to influence price.



In [ ]:
sns.relplot(y="price", x="water_distance", data=kc_data);

In [ ]:
sns.relplot(y="sqft_price", x="water_distance", data=kc_data);

These charts again support our hypothesis: price tends to decrease as distance from the water increases, and the price per square foot also falls.

The map offers an additional sanity check. Houses close to the water should have small `water_distance` values, and that is exactly what we see.



In [ ]:
fig = px.scatter_map(
    kc_data.query("water_distance <= 5"),
    lat="lat",
    lon="long",
    hover_name="id",
    hover_data=["sqft_price", "sqft_living", "zipcode", "floors"],
    size="sqft_price",
    color="water_distance",
    color_continuous_scale=["green", "yellow", "red"],
    color_discrete_sequence=["fuchsia"],
    zoom=8.4,
    height=400,
)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

# Modeling

Now that we understand the data better, we can start building a model to estimate the value of new houses.

Our goal is to predict a house price using information that would already be known before the sale.
Because we do not have truly future observations yet, we simulate that situation by splitting the existing dataset into a training set and a test set.

The training set is used to fit the model.
The test set acts like unseen future data and helps us estimate how well the model generalizes.

To create this split, we use scikit-learn's **train/test split**.
We specify:
- the input variables `X`
- the target variable `y`
- the proportion reserved for testing
- a `random_state` for reproducibility

The `random_state` matters because the rows are shuffled before the split. Without shuffling, an unlucky ordering of the data could bias the evaluation.
Using a fixed seed ensures that we can reproduce the same split later.

Before splitting, we also remove columns that would create data leakage or provide no useful predictive signal.

```mermaid
flowchart LR
    A[Select features X and target y] --> B[Remove leakage-prone columns]
    B --> C[Train/test split]
    C --> D[Train model]
    D --> E[Evaluate on test set]
    E --> F[Tune and regularize]
    F --> G[Inspect errors]
    G --> H[Save final model]
```



In [ ]:
# Remove columns that would cause data leakage or add no predictive value.
drop_lst = [
    "price",
    "sqft_price",
    "date",
    "delta_lat",
    "delta_long",
]

In [ ]:
# Keep all remaining variables as candidate features.
all_features = [x for x in kc_data.columns if x not in drop_lst]

In [ ]:
# `X` contains the descriptive variables defined above.
X = kc_data[all_features]

In [ ]:
# Define `y` as the target variable.
y = kc_data.price

In [ ]:
# Split the data into training and test sets. We reserve 30% for evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# Inspect the size of each split.
print("X_train (features for the model to learn from): ", X_train.shape)
print("y_train (labels for the model to learn from): ", y_train.shape)
print("X_test (features to test the model's accuracy against): ", X_test.shape)
print("y_test (labels to test the model's accuracy with): ", y_test.shape)

In [ ]:
# The training data is now shuffled rather than sorted by index.
X_train.head()

After the split, we can start modeling.
We continue with scikit-learn, where the overall workflow is usually the same:

- import the algorithm
- instantiate the model with its hyperparameters
- choose the feature columns
- fit the model with `.fit(X_train, y_train)`
- evaluate it on the test data

This repeated pattern makes it easy to compare several models in a structured way.



In [ ]:
# Instantiate the model.
model_lin_reg = LinearRegression()

Next, we decide which variables to pass to the model.
We begin with a very simple baseline that uses only one predictor: the house `grade`.



In [ ]:
# Select the variables passed to the model.
variables = ["grade"]

In [ ]:
# Train the model.
model_lin_reg.fit(X_train[variables], y_train)

In [ ]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test[variables], y_test))
        * (X_test.shape[0] - 1)
        / (X_test.shape[0] - len(variables) - 1),
        2,
    ),
)

**R²** measures the proportion of variance in the target variable that is explained by the model.
**Adjusted R²** is a related metric that also accounts for the number of predictors, which makes it more useful when comparing models of different complexity.

Interpretation:
- `1.0` means the model explains all observed variance
- `0.0` means the model explains none of it

In our case, using only `grade` explains about **43%** of the variance in price per square foot on the test set.
That is a useful start, but it suggests that additional variables may improve the model.

Let us inspect the available features again.



In [ ]:
# Show the names of the variables in the dataset.
X_train.columns

We can use more features to predict price per square foot.
The age of the house may also matter, so next we try a linear regression that includes both `grade` and age-related information.



In [ ]:
# Instantiate the model.
model_lin_reg = LinearRegression()

In [ ]:
# Select the variables passed to the model.
variables = ["grade", "last_known_change"]

In [ ]:
# Train the model.
model_lin_reg.fit(X_train[variables], y_train)

In [ ]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test[variables], y_test))
        * (X_test.shape[0] - 1)
        / (X_test.shape[0] - len(variables) - 1),
        2,
    ),
)

With this additional variable, the model explains about **48%** of the variance in price per square foot.



So far, we have only modeled linear relationships between the predictors and price.
However, some effects may be curved rather than straight.

One simple way to capture that is to add squared terms. Instead of fitting only:

$price = b \cdot grade + c$

we can also fit:

$price = a \cdot grade^{2} + b \cdot grade + c$

This is another form of feature engineering. We will apply it to the dataset and see whether the richer representation improves performance.



In [ ]:
# Create only second-order polynomial features (^2).
poly = PolynomialFeatures(2)

In [ ]:
# Create copies of the training and test data.
X_train_poly = X_train.copy()
X_test_poly = X_test.copy()

# Drop the `id` column.
X_train_poly = X_train_poly.drop(columns=["id"])
X_test_poly = X_test_poly.drop(columns=["id"])

In [ ]:
# Create transformed variables with `poly`.
X_train_sq = poly.fit_transform(X_train_poly)

# Apply the same transformation to the test data.
X_test_sq = poly.transform(X_test_poly)

In [ ]:
# Instantiate the model.
model_lin_reg = LinearRegression()

In [ ]:
# Train the model with the squared features.
model_lin_reg.fit(X_train_sq, y_train)

In [ ]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - model_lin_reg.score(X_test_sq, y_test))
        * (X_test_sq.shape[0] - 1)
        / (X_test_sq.shape[0] - X_test_sq.shape[1] - 1),
        2,
    ),
)

Adding squared features improved the model a little further.



Adjusted R² gives us one useful summary of model quality, but it is also important to inspect the errors directly.
A **residual analysis** can reveal systematic problems that a single score might hide.  

For easier interpretation, we calculate the percentage difference between the predicted and true prices.



In [ ]:
# Error analysis
# To inspect the model errors more closely, create a new DataFrame with the
# columns `price` (the true value), `lat`, and `long`.
y_predictions = model_lin_reg.predict(X_test_sq)
df_error = pd.DataFrame(y_test)
df_error["latitude"] = X_test["lat"]
df_error["longitude"] = X_test["long"]
df_error["id"] = X_test["id"]
df_error.head(2)

In [ ]:
# Reset the index before adding the predicted price as a new column.
df_error.reset_index(inplace=True, drop=True)
df_error.head(2)

In [ ]:
# Add the predicted price and calculate the difference.
df_error["price_prediction"] = y_predictions.round(2)
df_error["price_difference"] = (df_error["price_prediction"] - df_error["price"]).round(
    2
)
df_error["price_difference_percent"] = (
    (df_error["price_difference"] / df_error["price"]) * 100
).round(2)
df_error.head(2)

In [ ]:
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7,
    height=400,
)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

We can already see a few strong outliers in the residual plot, so it is worth inspecting the largest one more closely.



In [ ]:
df_error[
    df_error["price_difference_percent"] == df_error["price_difference_percent"].max()
]

In [ ]:
X_test[X_test["id"] == 9272202260]

Let us inspect these houses in more detail.
King County provides a helpful property lookup tool on [this page](https://localscape.property/#kingcountyassessor/My-Property), where you can search by parcel ID.

To inspect the outlier:
- change the selector in the top-left corner from `Address` to `Parcel ID`
- enter the `id` of the outlier

Under **Basic Property Characteristics** and in the **Aerial Images** view, we can see that there is no longer a house on the property. That explains why our model estimated a much higher sale value than the actual transaction suggests.



## Regularization and Hyperparameter Tuning

After adding squared features, the model has many more inputs.
Some of them may contribute useful signal, while others mainly capture noise.
If a model learns noise instead of general patterns, it becomes **overfitted**.

**Regularization** helps control this problem by shrinking coefficients and, in some cases, pushing some of them all the way to zero.
That allows the model to keep the more informative variables while reducing reliance on weaker ones.

Here we use **ElasticNet**, which combines L1 and L2 regularization.
You can read more about it [here](https://scikit-learn.org/stable/modules/linear_model.html#elastic-net).

To find a good amount of regularization, we test multiple hyperparameter values and compare the results with [GridSearch](https://scikit-learn.org/stable/modules/grid_search.html#grid-search) and [cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation).



In [ ]:
# Specify the hyperparameters to tune.
# `alpha` controls the amount of regularization.
param_grid = {
    "alpha": [
        0.1,
        0.5,
        1,
        5,
        10,
    ],
    "l1_ratio": [1, 0.5, 0],
}

# Instantiate the model.
elastic = ElasticNet(max_iter=50000, tol=0.2)

# Pass the model to a parameter grid with 5-fold cross-validation.
elastic_cv = GridSearchCV(elastic, param_grid, cv=5, verbose=True, n_jobs=-1)

# Train and optimize the model with grid search.
elastic_cv.fit(X_train_sq, y_train)

In [ ]:
# Display the best parameters found by the grid search.
elastic_cv.best_params_

In [ ]:
# Train the model with the best hyperparameters.
elastic = ElasticNet(max_iter=50000, tol=0.2, **elastic_cv.best_params_)

elastic.fit(X_train_sq, y_train)

In [ ]:
# Evaluate how well the model performs on the test data.
print(
    "Adj. R^2:",
    round(
        1
        - (1 - elastic.score(X_test_sq, y_test))
        * (X_test_sq.shape[0] - 1)
        / (X_test_sq.shape[0] - (X_test_sq.shape[1] - sum(elastic.coef_ == 0)) - 1),
        2,
    ),
)

Hyperparameter tuning and regularization improved the model again.  

As discussed above, regularization can set some regression coefficients to zero, effectively removing less useful variables from the model.  

The following code displays part of the learned coefficient vector, where you can already see an example of a coefficient that was reduced to zero.



In [ ]:
# Display the first 5 learned coefficients of the linear regression.
elastic.coef_[0:5]

In [ ]:
# Error analysis
# To inspect the model errors more closely, create a new DataFrame with the
# columns `price` (the true value), `lat`, and `long`.
y_predictions = elastic.predict(X_test_sq)
df_error = pd.DataFrame(y_test)
df_error["latitude"] = X_test["lat"]
df_error["longitude"] = X_test["long"]
df_error["id"] = X_test["id"]
df_error.head(2)

In [ ]:
# Reset the index before adding the predicted price as a new column.
df_error.reset_index(inplace=True, drop=True)
df_error.head(2)

In [ ]:
# Add the predicted price and calculate the difference.
df_error["price_prediction"] = y_predictions.round(2)
df_error["price_difference"] = (df_error["price_prediction"] - df_error["price"]).round(
    2
)
df_error["price_difference_percent"] = (
    (df_error["price_difference"] / df_error["price"]) * 100
).round(2)
df_error.head(2)

In [ ]:
fig = px.scatter_map(
    df_error,
    lat="latitude",
    lon="longitude",
    hover_data=["price", "price_prediction", "id"],
    color="price_difference_percent",
    color_continuous_scale=["green", "yellow", "red"],
    zoom=7.7,
    height=400,
)
fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r": 0, "t": 0, "l": 0, "b": 0})
fig.show()

In [ ]:
df_error[
    df_error["price_difference_percent"] == df_error["price_difference_percent"].max()
]

In [ ]:
X_test[X_test["id"] == 5111400086]

Let us take a closer look at this house on [this page](https://localscape.property/#kingcountyassessor/My-Property).

This house sold in 2018 for $295,000. King County appraisal data also reports historical values, and under **Historical Value** we can see that the **Total Assessed Value** in 2014 was $212,000.
That means the sale price was still unusually low relative to the broader context, which helps explain why it stands out so strongly in the residual analysis.



## Save the Model

We have now trained a model that can predict house prices in King County.
The final step is to save the trained model with [skops](https://skops.readthedocs.io/en/stable/modules/classes.html#module-skops.io).

If you want to explore the optional application stretch goal, see [bonus_solution/](./bonus_solution/). It contains a small working FastAPI + Postgres + Docker example that is meant as a reference, not a required solution path.



In [ ]:
import os
import skops.io as sio

# Create the directory if it does not already exist.
os.makedirs("model", exist_ok=True)

with open("model/model.bin", "wb") as f_out:
    sio.dump(elastic, f_out)

### Conclusion

Despite the presence of outliers, we built a model that explains roughly **83%** of the variance in the target.  

The notebook also shows an important modeling lesson: **feature engineering, nonlinear terms, and regularization can all make a meaningful difference in predictive performance**.

